In [11]:
pip install s3fs

  Using cached s3fs-2026.2.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached aiobotocore-3.2.0-py3-none-any.whl.metadata (26 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiohttp-3.13.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached aioitertools-0.13.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached multidict-6.7.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.3 kB)
  Using cached wrapt-2.1.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (7.4 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached propcache-0.4.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached yarl-1.23.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached s3fs-2026.2.0-py3-none-any.whl (31 kB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
U

In [12]:
import pandas as pd
import s3fs
from datetime import date, timedelta

BUCKET = "diamond-dna"
PREFIX = "raw-data/statcast"

start_date = date(2025, 4, 1)
end_date   = date(2025, 4, 3)

def build_daily_paths(bucket, prefix, start_date, end_date):
    paths = []
    current = start_date
    while current <= end_date:
        ds = current.strftime("%Y-%m-%d")
        year = current.year
        key = f"{prefix}/year={year}/date={ds}/statcast_pitches.parquet"
        paths.append(f"s3://{bucket}/{key}")
        current += timedelta(days=1)
    return paths

paths = build_daily_paths(BUCKET, PREFIX, start_date, end_date)

fs = s3fs.S3FileSystem()  # uses your AWS creds

# Tell pandas/pyarrow to use the S3 filesystem
df = pd.read_parquet(paths, filesystem=fs)

df.shape, df.head()

((9450, 120),
   pitch_type   game_date  release_speed  release_pos_x  release_pos_z  \
 0         FC  2025-04-01           90.9          -2.37           5.41   
 1         FS  2025-04-01           86.1          -2.48           5.33   
 2         FS  2025-04-01           86.2          -2.36           5.52   
 3         FS  2025-04-01           86.8          -2.34           5.45   
 4         FC  2025-04-01           90.6          -2.21            5.5   
 
        player_name  batter  pitcher     events      description  ...  \
 0  Eovaldi, Nathan  682829   543135  field_out    hit_into_play  ...   
 1  Eovaldi, Nathan  682829   543135       None     blocked_ball  ...   
 2  Eovaldi, Nathan  682829   543135       None         foul_tip  ...   
 3  Eovaldi, Nathan  682829   543135       None  swinging_strike  ...   
 4  Eovaldi, Nathan  682829   543135       None             ball  ...   
 
    api_break_x_arm  api_break_x_batter_in  arm_angle  attack_angle  \
 0            -0.03          